In [ ]:
import torchvision
from torchvision import transforms

# EX01
transform = transforms.Compose([transforms.RandomRotation(15), transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

# EX02
transform_aumentado = transforms.Compose([
    transforms.RandomAffine(translate=(0.1, 0.1), scale=(0.9, 1.1), degrees=15),
    transforms.RandomPerspective(),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# # EX04
# - Baseline: O modelo original, sem aumento de dados explícito (além da normalização).
transform_baseline = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
# - Básico: Usando RandomAffine (rotação, translação).
transform_basico = transforms.Compose([
    transforms.RandomAffine(translate=(0.1, 0.1), degrees=15),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
# - Avançado: Usando TrivialAugmentWide combinado com RandomErasing.
transform_avancado = transforms.Compose([
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    transforms.RandomErasing() # p=0.5 por padrão
])

In [ ]:
# MNIST dataset
root_path = '/home/storopoli/Downloads' # mude isso no Colab se necessário

# Train/Test Datasets
train_dataset_aumentado = torchvision.datasets.MNIST(root=root_path, train=True, transform=transform_aumentado, download=True)
train_dataset_baseline = torchvision.datasets.MNIST(root=root_path, train=True, transform=transform_baseline, download=True)
train_dataset_basico = torchvision.datasets.MNIST(root=root_path, train=True, transform=transform_basico, download=True)
train_dataset_avancado = torchvision.datasets.MNIST(root=root_path, train=True, transform=transform_avancado, download=True)

test_dataset = torchvision.datasets.MNIST(root=root_path, train=False, transform=transform_aumentado)

In [ ]:
from torch.utils.data import DataLoader

batch_size=32

train_loader_aumentado = DataLoader(dataset=train_dataset_aumentado, batch_size=batch_size, shuffle=True)
train_loader_baseline = DataLoader(dataset=train_dataset_baseline, batch_size=batch_size, shuffle=True)
train_loader_basico = DataLoader(dataset=train_dataset_basico, batch_size=batch_size, shuffle=True)
train_loader_avancado = DataLoader(dataset=train_dataset_avancado, batch_size=batch_size, shuffle=True)

test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import torch.nn as nn

In [ ]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.fc1 = nn.Sequential(
            nn.Linear(7 * 7 * 64, 1000),
            nn.ReLU())
        self.fc2 = nn.Linear(1000, 10)
    
    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc1(out)
        out = self.fc2(out)
        return out

# Instancia o Model()
model = ConvNet()

print(model)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

count_parameters(model)

In [ ]:
from torch.optim import Adam, SGD 

# Hiperparâmetros
loss_fn = nn.CrossEntropyLoss()
learning_rate = 0.001
epochs = 6

# Instânciar o Otimizador Adam
optimizer = Adam(model.parameters(), lr=learning_rate)



In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
def treinar_modelo(model, train_loader, loss_fn, optimizer, device, epochs):
    
    model.to(device)
    
    # Treinar o Modelo
    total_step = len(train_loader) # quantos batches eu tenho

    # Listas vazias
    loss_list = []
    acc_list = []
    epoch_loss_list = []
    epoch_acc_list = []

    for epoch in range(epochs):
        epoch_loss = 0
        epoch_acc = 0
        for i, (images, labels) in enumerate(train_loader):
            
            images, labels = images.to(device), labels.to(device)

            # Gera a propagação (feed forward)
            outputs = model(images)

            # Calcula a função-custo
            loss = loss_fn(outputs, labels)
            loss_list.append(loss.item())
            epoch_loss += loss.item()

            # Retro-propagação (Backprop) e a otimização com Adam
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Acurácia
            total = labels.size(0)
            _, predicted = torch.max(outputs.data, 1)
            correct = (predicted == labels).sum().item()
            acc_list.append(correct / total)
            epoch_acc += correct / total
            if (i + 1) % 100 == 0:
                print(f"Época [{epoch+1}/{epochs}], Step [{i+1}/{total_step}], Custo: {round(loss.item(), 3)}, Acurácia: {round((correct / total) * 100, 3)}")


        epoch_loss_list.append(epoch_loss / total_step)
        epoch_acc_list.append(epoch_acc / total_step)
    return epoch_loss_list, epoch_acc_list

In [ ]:
#treinar_modelo(model, train_loader_aumentado, loss_fn, optimizer, device, epochs)

model_baseline = ConvNet()
optimizer = Adam(model_baseline.parameters(), lr=learning_rate)
loss_baseline, acc_baseline = treinar_modelo(model_baseline, train_loader_baseline, loss_fn, optimizer, device, epochs)

model_basico = ConvNet()
optimizer = Adam(model_basico.parameters(), lr=learning_rate)
loss_basico, acc_basico = treinar_modelo(model_basico, train_loader_basico, loss_fn, optimizer, device, epochs)

model_avancado = ConvNet()
optimizer = Adam(model_avancado.parameters(), lr=learning_rate)
loss_avancado, acc_avancado = treinar_modelo(model_avancado, train_loader_avancado, loss_fn, optimizer, device, epochs)

In [ ]:
import matplotlib.pyplot as plt

# ======== EX 3: Plot dos gráficos de perda e acurácia ========
plt.figure(figsize=(10, 6))

plt.subplot(1, 2, 1)
plt.plot(loss_baseline, label="Baseline")
plt.plot(loss_basico, label="Básico")
plt.plot(loss_avancado, label="Avançado")
plt.xlabel("Épocas")
plt.ylabel("Loss médio por época")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(acc_baseline, label="Baseline")
plt.plot(acc_basico, label="Básico")
plt.plot(acc_avancado, label="Avançado")
plt.xlabel("Épocas")
plt.ylabel("Acurácia média por época")
plt.legend()

plt.suptitle("Velocidade de convergência")
plt.show()

In [ ]:
model.eval() # coloca o modelo em modo de avaliação (sem calcular gradientes)
model_baseline.eval()
model_basico.eval()
model_avancado.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:

        # Move tensores para o dispositivo configurado (CPU ou GPU)
        images, labels = images.to(device), labels.to(device)

        # Feed-forward com as imagens de teste
        outputs = model_basico(images)
        
        # gera predições usando a função max()
        _, predicted = torch.max(outputs.data, 1)
        
        # Acumula total e corretas
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    print(f"Acurácia do Modelo em 10k imagens de teste: {round((correct / total) * 100, 3)}")